## 1. Import thư viện cho EDA & Regression Modeling

In [11]:
import kagglehub

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error
from sklearn.impute import KNNImputer

In [12]:
path = kagglehub.dataset_download("nelgiriyewithana/global-weather-repository")
print("Path to dataset files:", path)

Path to dataset files: C:\Users\Admin\.cache\kagglehub\datasets\nelgiriyewithana\global-weather-repository\versions\1022


In [13]:
df = pd.read_csv(f"{path}/GlobalWeatherRepository.csv")

In [14]:
df

,country,location_name,latitude,longitude,timezone,last_updated_epoch,last_updated,temperature_celsius,temperature_fahrenheit,condition_text,...,air_quality_PM2.5,air_quality_PM10,air_quality_us-epa-index,air_quality_gb-defra-index,sunrise,sunset,moonrise,moonset,moon_phase,moon_illumination
0,Afghanistan,Kabul,34.5200,69.1800,Asia/Kabul,1715849100,2024-05-16 13:15,26.6,79.8,Partly Cloudy,...,8.4,26.6,1,1,04:50 AM,06:50 PM,12:12 PM,01:11 AM,Waxing Gibbous,55
1,Albania,Tirana,41.3300,19.8200,Europe/Tirane,1715849100,2024-05-16 10:45,19.0,66.2,Partly cloudy,...,1.1,2.0,1,1,05:21 AM,07:54 PM,12:58 PM,02:14 AM,Waxing Gibbous,55
2,Algeria,Algiers,36.7600,3.0500,Africa/Algiers,1715849100,2024-05-16 09:45,23.0,73.4,Sunny,...,10.4,18.4,1,1,05:40 AM,07:50 PM,01:15 PM,02:14 AM,Waxing Gibbous,55
3,Andorra,Andorra La Vella,42.5000,1.5200,Europe/Andorra,1715849100,2024-05-16 10:45,6.3,43.3,Light drizzle,...,0.7,0.9,1,1,06:31 AM,09:11 PM,02:12 PM,03:31 AM,Waxing Gibbous,55
4,Angola,Luanda,-8.8400,13.2300,Africa/Luanda,1715849100,2024-05-16 09:45,26.0,78.8,Partly cloudy,...,183.4,262.3,5,10,06:12 AM,05:55 PM,01:17 PM,12:38 AM,Waxing Gibbous,55
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
154551,Venezuela,Caracas,10.5000,-66.9167,America/Caracas,1784613600,2026-07-21 02:00,18.2,64.8,Light rain shower,...,7.0,18.1,1,1,06:14 AM,06:53 PM,12:39 PM,Does not set today,First Quarter,47
154552,Vietnam,Hanoi,21.0333,105.8500,Asia/Bangkok,1784613600,2026-07-21 13:00,33.2,91.8,Patchy rain nearby,...,71.2,72.2,4,10,05:26 AM,06:39 PM,11:55 AM,11:27 PM,First Quarter,42
154553,Yemen,Sanaa,15.3547,44.2067,Asia/Aden,1784613600,2026-07-21 09:00,25.3,77.5,Sunny,...,19.5,56.2,2,2,05:42 AM,06:36 PM,12:04 PM,11:49 PM,First Quarter,44
154554,Zambia,Lusaka,-15.4167,28.2833,Africa/Lusaka,1784613600,2026-07-21 08:00,18.1,64.6,Sunny,...,27.8,29.4,2,3,06:33 AM,05:53 PM,11:35 AM,Does not set today,First Quarter,45


## 2. Khám phá & làm sạch dữ liệu

Kiểm tra kiểu dữ liệu, giá trị thiếu, và loại bỏ các cột trùng đơn vị / rò rỉ dữ liệu (leakage) so với target `temperature_celsius`.

In [17]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 154556 entries, 0 to 154555
Data columns (total 41 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   country                       154556 non-null  str    
 1   location_name                 154556 non-null  str    
 2   latitude                      154556 non-null  float64
 3   longitude                     154556 non-null  float64
 4   timezone                      154556 non-null  str    
 5   last_updated_epoch            154556 non-null  int64  
 6   last_updated                  154556 non-null  str    
 7   temperature_celsius           154556 non-null  float64
 8   temperature_fahrenheit        154556 non-null  float64
 9   condition_text                154556 non-null  str    
 10  wind_mph                      154556 non-null  float64
 11  wind_kph                      154556 non-null  float64
 12  wind_degree                   154556 non-null  int64  


In [18]:
df.isna().sum()[df.isna().sum() > 0]   # xác nhận không có giá trị thiếu

Series([], dtype: int64)

In [19]:
df.describe()

,latitude,longitude,last_updated_epoch,temperature_celsius,temperature_fahrenheit,wind_mph,wind_kph,wind_degree,pressure_mb,pressure_in,...,gust_kph,air_quality_Carbon_Monoxide,air_quality_Ozone,air_quality_Nitrogen_dioxide,air_quality_Sulphur_dioxide,air_quality_PM2.5,air_quality_PM10,air_quality_us-epa-index,air_quality_gb-defra-index,moon_illumination
count,154556.000000,154556.000000,1.545560e+05,154556.000000,154556.000000,154556.000000,154556.000000,154556.000000,154556.000000,154556.000000,...,154556.000000,154556.000000,154556.000000,154556.000000,154556.000000,154556.000000,154556.000000,154556.000000,154556.000000,154556.000000
mean,19.246216,21.875444,1.750248e+09,21.375925,70.478423,7.933044,12.770765,169.636352,1014.053120,29.944411,...,18.080687,432.805146,57.752356,14.356608,9.841722,23.268821,46.721385,1.666160,2.529064,49.815950
std,24.396293,65.776434,1.985835e+07,9.473777,17.052659,6.984442,11.236892,103.526436,9.932215,0.293242,...,13.494943,721.276273,30.752822,22.771546,33.701481,35.376485,143.835462,0.923715,2.392493,35.076452
min,-41.300000,-175.200000,1.715849e+09,-29.800000,-21.600000,2.200000,3.600000,1.000000,947.000000,27.960000,...,3.600000,-9999.000000,0.000000,0.000000,-9999.000000,0.168000,-1848.150000,1.000000,1.000000,0.000000
25%,4.050300,-6.836100,1.733135e+09,16.100000,61.000000,3.800000,6.100000,81.000000,1010.000000,29.830000,...,10.000000,184.850000,38.000000,1.750000,1.100000,6.850000,9.650000,1.000000,1.000000,15.000000
50%,17.250000,21.433300,1.750238e+09,23.700000,74.700000,6.700000,10.800000,163.000000,1014.000000,29.930000,...,15.100000,277.000000,54.400000,5.550000,2.400000,13.505000,19.120000,1.000000,2.000000,50.000000
75%,40.400000,49.882200,1.767338e+09,27.900000,82.200000,10.700000,17.300000,256.000000,1018.000000,30.060000,...,24.000000,436.600000,73.000000,16.350000,7.770000,26.640000,39.960000,2.000000,3.000000,85.000000
max,65.300000,179.220000,1.784614e+09,79.300000,174.700000,1841.200000,2963.200000,360.000000,3006.000000,88.770000,...,2970.400000,38879.398000,480.700000,427.700000,521.330000,1614.100000,6037.290000,6.000000,10.000000,100.000000


In [25]:
# Loại các cột trùng đơn vị (giữ 1 cột/đại lượng, giữ đơn vị celsius và km)
drop_cols = [
    'temperature_fahrenheit',        # trùng temperature_celsius (target)
    'feels_like_celsius', 'feels_like_fahrenheit',  # tính trực tiếp từ temperature -> leakage
    'wind_mph', 'gust_mph',          # trùng wind_kph / gust_kph
    'pressure_in',                   # trùng pressure_mb
    'visibility_miles',              # trùng visibility_km
]
df_model = df.drop(columns=drop_cols)

# Biến đổi last_updated thành int
df_model['last_updated'] = pd.to_datetime(df_model['last_updated'])
df_model['month'] = df_model['last_updated'].dt.month
df_model['hour'] = df_model['last_updated'].dt.hour

df_model[['last_updated', 'month', 'hour']]

,last_updated,month,hour
0,2024-05-16 13:15:00,5,13
1,2024-05-16 10:45:00,5,10
2,2024-05-16 09:45:00,5,9
3,2024-05-16 10:45:00,5,10
4,2024-05-16 09:45:00,5,9
...,...,...,...
154551,2026-07-21 02:00:00,7,2
154552,2026-07-21 13:00:00,7,13
154553,2026-07-21 09:00:00,7,9
154554,2026-07-21 08:00:00,7,8


## 2b. Chuẩn hoá tên quốc gia & lọc chỉ Châu Á

Cột `country` chứa nhiều biến thể trùng lặp do lỗi dịch/nhập liệu (vd `Inde`→India, `Südkorea`→South Korea,
`Kyrghyzstan`→Kyrgyzstan, `火鸡`/`Турция`→Turkey...) — đúng vấn đề "chuẩn hoá cách biểu diễn dữ liệu" nêu ở Bài 02.
Cần chuẩn hoá các alias này trước khi lọc, nếu không sẽ bỏ sót một số dòng hợp lệ của các nước Châu Á.

In [ ]:
# Alias cho các biến thể tên quốc gia (đa ngôn ngữ / lỗi chính tả) xuất hiện trong dữ liệu
ASIA_ALIASES = {
    'Inde': 'India', 'Jemen': 'Yemen', 'Malásia': 'Malaysia', 'Südkorea': 'South Korea',
    'Turkménistan': 'Turkmenistan', 'Турция': 'Turkey', '火鸡': 'Turkey',
    'Saudi Arabien': 'Saudi Arabia', 'Kyrghyzstan': 'Kyrgyzstan',
}

# Danh sách quốc gia Châu Á (theo UN geoscheme, tên chuẩn tiếng Anh xuất hiện trong dữ liệu)
ASIA_COUNTRIES = {
    'Afghanistan', 'Armenia', 'Azerbaijan', 'Bahrain', 'Bangladesh', 'Bhutan', 'Brunei Darussalam',
    'Cambodia', 'China', 'Cyprus', 'Georgia', 'India', 'Indonesia', 'Iran', 'Iraq', 'Israel', 'Japan',
    'Jordan', 'Kazakhstan', 'Kuwait', 'Kyrgyzstan', "Lao People's Democratic Republic", 'Lebanon',
    'Malaysia', 'Maldives', 'Mongolia', 'Myanmar', 'Nepal', 'North Korea', 'Oman', 'Pakistan',
    'Philippines', 'Qatar', 'Saudi Arabia', 'Singapore', 'South Korea', 'Sri Lanka', 'Syria',
    'Tajikistan', 'Thailand', 'Timor-Leste', 'Turkey', 'Turkmenistan', 'United Arab Emirates',
    'Uzbekistan', 'Vietnam', 'Yemen',
}

df_model['country_clean'] = df_model['country'].replace(ASIA_ALIASES)
df_model = df_model[df_model['country_clean'].isin(ASIA_COUNTRIES)].reset_index(drop=True)

print('Số dòng sau khi lọc Châu Á:', len(df_model))
print('Số quốc gia:', df_model['country_clean'].nunique())
df_model[['country', 'country_clean', 'location_name']].head()

# PHẦN A — Dự đoán `temperature_celsius` (Regression, chỉ dữ liệu Châu Á)

## 3. Tương quan & chọn đặc trưng (feature selection)

Dùng hệ số Pearson để đánh giá mức độ liên hệ tuyến tính giữa từng biến số với `temperature_celsius`, loại các biến có |r| trong khoảng -0.3 đến 0.3 (tương quan yếu/không có, theo quy tắc trong Bài 05).

In [ ]:
candidate_features = [
    'latitude', 'humidity', 'cloud', 'pressure_mb', 'wind_kph',
    'uv_index', 'precip_mm', 'visibility_km', 'month', 'hour',
]

corr_target = df_model[candidate_features + ['temperature_celsius']].corr()['temperature_celsius'].drop('temperature_celsius')
corr_target.sort_values(key=abs, ascending=False)

In [ ]:
plt.figure(figsize=(9, 7))
sns.heatmap(df_model[candidate_features + ['temperature_celsius']].corr(), annot=True, fmt='.2f', cmap='RdBu', center=0)
plt.title('Ma trận tương quan Pearson với temperature_celsius (Châu Á)')
plt.tight_layout()
plt.show()

In [ ]:
# Chọn các đặc trưng có |r| > 0.3 với target làm tập feature cho mô hình
selected_features = corr_target[corr_target.abs() > 0.3].index.tolist()
print('Selected features:', selected_features)

X = df_model[selected_features]
y = df_model['temperature_celsius']

## 4. Simple Linear Regression (SLR) — baseline

Dùng đặc trưng có tương quan mạnh nhất làm baseline SLR, đánh giá bằng regression plot & residual plot.

In [ ]:
best_single_feature = corr_target.abs().idxmax()
print('Biến mạnh nhất:', best_single_feature, '| r =', corr_target[best_single_feature])

slr = LinearRegression()
X_slr = df_model[[best_single_feature]]
slr.fit(X_slr, y)
Yhat_slr = slr.predict(X_slr)

print('intercept:', slr.intercept_, '| coef:', slr.coef_)
print('R^2 (train, toàn bộ dữ liệu):', slr.score(X_slr, y))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sample = df_model.sample(3000, random_state=0)  # lấy mẫu để vẽ nhanh hơn với 154k dòng
sns.regplot(x=best_single_feature, y='temperature_celsius', data=sample, ax=axes[0], scatter_kws={'alpha': 0.3})
axes[0].set_title(f'Regression plot: temperature_celsius ~ {best_single_feature}')

sns.residplot(x=X_slr[best_single_feature], y=y, ax=axes[1])
axes[1].set_title('Residual plot (SLR)')
plt.tight_layout()
plt.show()

## 5. Multiple Linear Regression (MLR) + Train/Test Split

Dùng toàn bộ `selected_features`, đánh giá đúng cách trên tập test (không chỉ trên train) để tránh đánh giá sai do overfitting.

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)

mlr = LinearRegression()
mlr.fit(x_train, y_train)

yhat_train = mlr.predict(x_train)
yhat_test = mlr.predict(x_test)

results = {}
results['MLR'] = {
    'R2_train': mlr.score(x_train, y_train),
    'R2_test': mlr.score(x_test, y_test),
    'RMSE_test': mean_squared_error(y_test, yhat_test) ** 0.5,
}
print('MLR intercept:', mlr.intercept_)
print(pd.Series(mlr.coef_, index=selected_features).sort_values(key=abs, ascending=False))
results['MLR']

In [ ]:
plt.figure(figsize=(6, 5))
sns.residplot(x=yhat_test, y=y_test - yhat_test)
plt.axhline(0, color='red', linestyle='--')
plt.xlabel('Predicted temperature_celsius (test)')
plt.ylabel('Residual (y - yhat)')
plt.title('Residual plot (MLR, test set)')
plt.tight_layout()
plt.show()

## 6. Polynomial Regression

Nếu residual plot của MLR cho thấy dạng cong, thử mở rộng bằng đặc trưng đa thức bậc 2 (chuẩn hoá trước bằng `StandardScaler` theo best practice trong tài liệu).

In [ ]:
poly_pipe = Pipeline([
    ('scale', StandardScaler()),
    ('polynomial', PolynomialFeatures(degree=2, include_bias=False)),
    ('model', LinearRegression()),
])
poly_pipe.fit(x_train, y_train)

yhat_test_poly = poly_pipe.predict(x_test)
results['Polynomial (deg=2)'] = {
    'R2_train': poly_pipe.score(x_train, y_train),
    'R2_test': poly_pipe.score(x_test, y_test),
    'RMSE_test': mean_squared_error(y_test, yhat_test_poly) ** 0.5,
}
results['Polynomial (deg=2)']

## 7. Ridge Regression (Regularization)

Các đặc trưng khí tượng (áp suất, độ ẩm, mây, gió) có thể đa cộng tuyến — dùng Ridge + `GridSearchCV` để tìm alpha tối ưu, kiểm tra Ridge có cải thiện so với Linear/Polynomial thường không.

In [ ]:
ridge_pipe = Pipeline([
    ('scale', StandardScaler()),
    ('polynomial', PolynomialFeatures(degree=2, include_bias=False)),
    ('model', Ridge()),
])

param_grid = {'model__alpha': np.logspace(-3, 4, 60)}
grid = GridSearchCV(ridge_pipe, param_grid, cv=4, scoring='r2')
grid.fit(x_train, y_train)

best_ridge = grid.best_estimator_
yhat_test_ridge = best_ridge.predict(x_test)

results['Ridge (best alpha)'] = {
    'R2_train': best_ridge.score(x_train, y_train),
    'R2_test': best_ridge.score(x_test, y_test),
    'RMSE_test': mean_squared_error(y_test, yhat_test_ridge) ** 0.5,
}
print('Best alpha:', grid.best_params_['model__alpha'])
results['Ridge (best alpha)']

## 8. So sánh mô hình

Bảng tổng hợp R²(train/test) và RMSE(test) của các mô hình đã thử — mô hình tốt là mô hình có R² test cao **và** không chênh lệch quá nhiều so với R² train (tránh overfitting).

In [ ]:
results['SLR'] = {
    'R2_train': slr.score(X_slr, y),
    'R2_test': np.nan,  # SLR ở trên fit trên toàn bộ dữ liệu để minh hoạ baseline, không qua train/test split
    'RMSE_test': np.nan,
}

results_df = pd.DataFrame(results).T[['R2_train', 'R2_test', 'RMSE_test']]
results_df.sort_values('R2_test', ascending=False)

# PHẦN B — Dự đoán `humidity` (chỉ số thời tiết dạng số, Regression, chỉ dữ liệu Châu Á)

Áp dụng lại đúng quy trình của Phần A (tương quan → SLR → MLR → Polynomial → Ridge) nhưng đổi target sang `humidity` (độ ẩm, %).
`precip_mm` không được chọn làm target vì ~67% giá trị bằng 0 (zero-inflated) — không phù hợp cho hồi quy tuyến tính thông thường.

## Tương quan & chọn đặc trưng cho `humidity`

In [ ]:
candidate_features_b = [
    'latitude', 'temperature_celsius', 'cloud', 'pressure_mb', 'wind_kph',
    'uv_index', 'precip_mm', 'visibility_km', 'month', 'hour',
]

corr_target_b = df_model[candidate_features_b + ['humidity']].corr()['humidity'].drop('humidity')
corr_target_b.sort_values(key=abs, ascending=False)

In [ ]:
plt.figure(figsize=(9, 7))
sns.heatmap(df_model[candidate_features_b + ['humidity']].corr(), annot=True, fmt='.2f', cmap='RdBu', center=0)
plt.title('Ma trận tương quan Pearson với humidity (Châu Á)')
plt.tight_layout()
plt.show()

In [ ]:
selected_features_b = corr_target_b[corr_target_b.abs() > 0.3].index.tolist()
print('Selected features:', selected_features_b)

X_b = df_model[selected_features_b]
y_b = df_model['humidity']

## SLR baseline cho `humidity`

In [ ]:
best_single_feature_b = corr_target_b.abs().idxmax()
print('Biến mạnh nhất:', best_single_feature_b, '| r =', corr_target_b[best_single_feature_b])

slr_b = LinearRegression()
X_slr_b = df_model[[best_single_feature_b]]
slr_b.fit(X_slr_b, y_b)

print('intercept:', slr_b.intercept_, '| coef:', slr_b.coef_)
print('R^2 (train, toàn bộ dữ liệu):', slr_b.score(X_slr_b, y_b))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sample_b = df_model.sample(3000, random_state=0)
sns.regplot(x=best_single_feature_b, y='humidity', data=sample_b, ax=axes[0], scatter_kws={'alpha': 0.3})
axes[0].set_title(f'Regression plot: humidity ~ {best_single_feature_b}')

sns.residplot(x=X_slr_b[best_single_feature_b], y=y_b, ax=axes[1])
axes[1].set_title('Residual plot (SLR, humidity)')
plt.tight_layout()
plt.show()

## MLR + Train/Test Split cho `humidity`

In [ ]:
x_train_b, x_test_b, y_train_b, y_test_b = train_test_split(X_b, y_b, test_size=0.3, random_state=0)

mlr_b = LinearRegression()
mlr_b.fit(x_train_b, y_train_b)

yhat_test_b = mlr_b.predict(x_test_b)

results_b = {}
results_b['MLR'] = {
    'R2_train': mlr_b.score(x_train_b, y_train_b),
    'R2_test': mlr_b.score(x_test_b, y_test_b),
    'RMSE_test': mean_squared_error(y_test_b, yhat_test_b) ** 0.5,
}
print('MLR intercept:', mlr_b.intercept_)
print(pd.Series(mlr_b.coef_, index=selected_features_b).sort_values(key=abs, ascending=False))
results_b['MLR']

In [ ]:
plt.figure(figsize=(6, 5))
sns.residplot(x=yhat_test_b, y=y_test_b - yhat_test_b)
plt.axhline(0, color='red', linestyle='--')
plt.xlabel('Predicted humidity (test)')
plt.ylabel('Residual (y - yhat)')
plt.title('Residual plot (MLR, humidity, test set)')
plt.tight_layout()
plt.show()

## Polynomial Regression cho `humidity`

In [ ]:
poly_pipe_b = Pipeline([
    ('scale', StandardScaler()),
    ('polynomial', PolynomialFeatures(degree=2, include_bias=False)),
    ('model', LinearRegression()),
])
poly_pipe_b.fit(x_train_b, y_train_b)

yhat_test_poly_b = poly_pipe_b.predict(x_test_b)
results_b['Polynomial (deg=2)'] = {
    'R2_train': poly_pipe_b.score(x_train_b, y_train_b),
    'R2_test': poly_pipe_b.score(x_test_b, y_test_b),
    'RMSE_test': mean_squared_error(y_test_b, yhat_test_poly_b) ** 0.5,
}
results_b['Polynomial (deg=2)']

## Ridge Regression cho `humidity`

In [ ]:
ridge_pipe_b = Pipeline([
    ('scale', StandardScaler()),
    ('polynomial', PolynomialFeatures(degree=2, include_bias=False)),
    ('model', Ridge()),
])

param_grid_b = {'model__alpha': np.logspace(-3, 4, 60)}
grid_b = GridSearchCV(ridge_pipe_b, param_grid_b, cv=4, scoring='r2')
grid_b.fit(x_train_b, y_train_b)

best_ridge_b = grid_b.best_estimator_
yhat_test_ridge_b = best_ridge_b.predict(x_test_b)

results_b['Ridge (best alpha)'] = {
    'R2_train': best_ridge_b.score(x_train_b, y_train_b),
    'R2_test': best_ridge_b.score(x_test_b, y_test_b),
    'RMSE_test': mean_squared_error(y_test_b, yhat_test_ridge_b) ** 0.5,
}
print('Best alpha:', grid_b.best_params_['model__alpha'])
results_b['Ridge (best alpha)']

## So sánh mô hình cho `humidity`

In [30]:
results_b['SLR'] = {
    'R2_train': slr_b.score(X_slr_b, y_b),
    'R2_test': np.nan,
    'RMSE_test': np.nan,
}

results_df_b = pd.DataFrame(results_b).T[['R2_train', 'R2_test', 'RMSE_test']]
results_df_b.sort_values('R2_test', ascending=False)

,R2_train,R2_test,RMSE_test
Ridge (best alpha),0.522504,0.520909,16.614446
Polynomial (deg=2),0.522527,0.520908,16.614458
MLR,0.488889,0.491504,17.116722
SLR,0.337259,NaN,NaN
